# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 34 · Route shape and directional persistence

**Round 16: six core channels and six extensions. No model fitting in this notebook.**

Three-, five- and ten-frame contiguous observed paths: current-heading displacement, path efficiency, turn accumulation and step-direction consistency. Durations are 0.2, 0.4 and 0.9 seconds. These are representation hypotheses, not new raw signals.

The current source-test receipt is created before preflight. Existing valid tests are reused; old models are not retrained.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json, sys, os, subprocess
import plotly.io as pio
import plotly.graph_objects as go
KIT = Path('/home/sagemaker-user/nfl_feature_rounds16_17')
ROUND = 16
OUT = Path(f'/home/sagemaker-user/nfl-feature-round{ROUND}-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space; do not create another environment.')
for _name in [p.stem for p in KIT.glob('*.py')]:
    _loaded = sys.modules.get(_name)
    _file = getattr(_loaded, '__file__', None)
    if _file and Path(_file).resolve().parent != KIT.resolve():
        raise RuntimeError('Restart this kernel: a different kit owns '+_name)
previous = globals().get('_NFL_ACTIVE_KIT')
if previous and previous != str(KIT):
    raise RuntimeError('Restart this kernel before changing kits.')
_NFL_ACTIVE_KIT = str(KIT)
os.chdir(KIT)
if str(KIT) not in sys.path: sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
PROCEED = False

def run(stage, *options):
    if stage in ('prepare','runtime','profile','train','evaluate','replay') and not PROCEED:
        print('REVIEW PENDING: no scientific stage executed. Return the readiness report.')
        return False
    command = [str(PY),str(KIT/'run_round.py'),stage,'--round',str(ROUND),*options]
    # The existing launcher enforces its stage cap, locks and worker-group shutdown.
    process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                               text=True,bufsize=1,cwd=KIT)
    try:
        for line in process.stdout: print(line,end='')
        result = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            raise RuntimeError('Launcher did not exit promptly; stop here and inspect its log. Do not restart concurrently.')
        raise
    if result:
        raise RuntimeError(f'{stage} stopped ({result}). Use the report cell; do not rerun unchanged failures.')
    return True

def show(fig, name):
    visuals.save(fig,OUT,name).show()

def collect():
    completed = subprocess.run([str(PY),'/home/sagemaker-user/nfl_workspace.zip','bundle'],cwd=KIT)
    if completed.returncode: raise RuntimeError('Report collection stopped; return the printed diagnostic.')

def history_plot(which, roles=False):
    summary = visuals.read(OUT.parent/f'nfl-feature-round{which}-results'/'summary.json')
    fig = go.Figure()
    if roles:
        for arm in ('mask','core','full','preserved_tree'):
            rows=[x for x in summary['slices'] if x['arm']==arm and x['slice'].startswith('role_')]
            fig.add_bar(name=arm,x=[x['slice'] for x in rows],y=[x['rmse'] for x in rows])
        fig.update_layout(barmode='group')
    else:
        names=list(summary['metrics'])
        fig.add_bar(x=names,y=[summary['metrics'][x]['rmse'] for x in names])
    return visuals.style(fig,f'Completed Round {which} — reused internal games, not Kaggle',
                         'Role code: 0 receiver, 1 coverage' if roles else 'Arm','Coordinate RMSE (yards)')


## Evidence and training coverage
These plots use completed Rounds 14–15 and the existing label audit. All six within-round feature gates failed. Mask-arm comparisons between rounds do not isolate candidate values. The final historical labels have not been independently re-audited by this package.

In [ ]:
show(history_plot(14), 'reviewed_metrics')
show(history_plot(14, roles=True), 'reviewed_roles')
show(visuals.coverage(OUT.parent/'nfl-feature-round14-results'/'label_readiness.json'),'training_label_coverage')

## 1. Run or reuse the shared source tests
The prior preflight failure was a missing `tests_receipt.json`. This cell creates it with the existing test runner, not a fabricated JSON file. Do not rerun failed tests unchanged.

In [ ]:
from readiness import require_tests
args = SimpleNamespace(kit=KIT, out=OUT, round_no=ROUND)
receipt = OUT.parent/'nfl-feature-round16-results'/'tests_receipt.json'
if not receipt.exists():
    print('Running the missing shared source tests once; no private model fit.')
    run('tests')
verified = require_tests(args)  # Failed or source-mismatched evidence is never treated as a pass.
print(json.dumps({'status':verified['status'],'tests_run':verified['tests_run'],
                  'failures':verified['failures'],'errors':verified['errors'],
                  'skipped':verified['skipped'],'shared_by_rounds':[16,17]},indent=2))


## 2. Parent/data preflight
Verifies the current source and frozen artifact chain without scoring validation. A failure is a stop; use the report cell.

In [ ]:
run('preflight')

## 3. Thirty-two training plays
Preserve real frame gaps and the exact historical motion representation. No target arrays are unpacked. Inspect support and magnitude; do not tune constants after inspection.

In [ ]:
run('smoke')
show(visuals.support(OUT),'candidate_support')
show(visuals.magnitude(OUT),'candidate_magnitudes')

## 4. Save and return evidence
Run the other research notebook too (34 then 36). Do not run ablation notebooks 35/37 before this new readiness evidence is reviewed. Save with Ctrl+S, close and reopen to check all five Plotly outputs. The single return filename stays `nfl_workspace_report.zip`.

In [ ]:
run('report')
collect()
print('/home/sagemaker-user/nfl_workspace_report.zip')